# Subsample data (2500 fps to 1000 fps)

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import resample

# ── Directories ───────────────────────────────────────────────────────
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled_1000fps"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ORIG_FPS = 2500
TARGET_FPS = 1000

# ── Find files ────────────────────────────────────────────────────────
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv"))
if not csv_files:
    raise FileNotFoundError(f"No SMOOTH CSV files found in {INPUT_DIR}")

for path in csv_files:
    fname = os.path.basename(path)
    print("Resampling:", fname)

    df = pd.read_csv(path)

    n_out = int(len(df) * TARGET_FPS / ORIG_FPS)

    sub_df = pd.DataFrame()
    sub_df["frame"] = np.arange(n_out)

    for col in df.columns[1:]:  # skip frame
        sig = df[col].values

        if np.all(np.isnan(sig)):
            sub_df[col] = np.full(n_out, np.nan)
            continue

        isnan = np.isnan(sig)
        out = np.full(n_out, np.nan)

        i = 0
        while i < len(sig):
            if not isnan[i]:
                j = i
                while j < len(sig) and not isnan[j]:
                    j += 1

                seg = sig[i:j]

                if len(seg) > 3:
                    res = resample(seg, int(len(seg) * TARGET_FPS / ORIG_FPS))
                else:
                    # fallback interpolation
                    x_old = np.arange(len(seg))
                    x_new = np.linspace(0, len(seg)-1,
                                        int(len(seg) * TARGET_FPS / ORIG_FPS))
                    res = np.interp(x_new, x_old, seg)

                # placement in output
                out_i = int(i * TARGET_FPS / ORIG_FPS)
                out_j = out_i + len(res)

                out[out_i:out_j] = res
                i = j
            else:
                i += 1

        sub_df[col] = out

    out_name = fname.replace("_SMOOTH.csv", "_SUB_1000fps.csv")
    sub_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

    print(f"  → Saved {out_name} ({len(sub_df)} frames)")

print("\n Done: Resampled to 1000 fps")

Resampling: Trial2_180rpmxyzpts_SMOOTH.csv
  → Saved Trial2_180rpmxyzpts_SUB_1000fps.csv (320 frames)
Resampling: Trial2_200rpmxyzpts_SMOOTH.csv
  → Saved Trial2_200rpmxyzpts_SUB_1000fps.csv (94 frames)
Resampling: Trial3_180rpmxyzpts_SMOOTH.csv
  → Saved Trial3_180rpmxyzpts_SUB_1000fps.csv (94 frames)
Resampling: Trial4_400rpmxyzpts_SMOOTH.csv
  → Saved Trial4_400rpmxyzpts_SUB_1000fps.csv (178 frames)
Resampling: Trial5_180rpmxyzpts_SMOOTH.csv
  → Saved Trial5_180rpmxyzpts_SUB_1000fps.csv (260 frames)
Resampling: Trial5_400rpmxyzpts_SMOOTH.csv
  → Saved Trial5_400rpmxyzpts_SUB_1000fps.csv (270 frames)
Resampling: Trial7_400rpmxyzpts_SMOOTH.csv
  → Saved Trial7_400rpmxyzpts_SUB_1000fps.csv (270 frames)

✅ Done: Resampled to 1000 fps


# Turning angle (center-based) 

In [1]:
import os
import glob
import numpy as np
import pandas as pd

# ── Config ─────────────────────────────────────────────────────────────
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled_1000fps"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based_1000fps"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS = 1000
DT  = 1 / FPS

# ── Helpers ───────────────────────────────────────────────────────────
def normalize(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0] = 1
    return v / n

def turning_angle_3d(v_in, v_out):
    dot   = np.sum(v_in * v_out, axis=1)
    cross = np.linalg.norm(np.cross(v_in, v_out), axis=1)
    return np.degrees(np.arctan2(cross, dot))

def turning_angle_2d(v_in, v_out):
    dot   = np.sum(v_in * v_out, axis=1)
    cross = v_in[:, 0]*v_out[:, 1] - v_in[:, 1]*v_out[:, 0]
    return np.degrees(np.arctan2(cross, dot))

# ── Process ───────────────────────────────────────────────────────────
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SUB_1000fps.csv"))

for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace("_SUB_1000fps.csv", "")
    print(f"Processing: {fname}")

    df = pd.read_csv(path)

    center = df[["center_X","center_Y","center_Z"]].values
    frames_all = df["frame"].values

    # ── Apply SAME mask to everything ─────────────────────────────────
    valid_mask = ~np.all(np.isnan(center), axis=1)

    center = center[valid_mask]
    frames_all = frames_all[valid_mask]

    if len(center) < 3:
        print("  ⚠ Skipping (too few valid points)")
        continue

    # ── CENTERED VECTORS (t-1, t, t+1) ────────────────────────────────
    v_in  = center[1:-1] - center[:-2]
    v_out = center[2:]   - center[1:-1]

    v_in  = normalize(v_in)
    v_out = normalize(v_out)

    # ── Turning angles ────────────────────────────────────────────────
    turn_3d = turning_angle_3d(v_in, v_out)

    turn_xy_raw = turning_angle_2d(
        normalize(v_in[:, :2]),
        normalize(v_out[:, :2])
    )

    turn_yz_raw = turning_angle_2d(
        normalize(v_in[:, 1:]),
        normalize(v_out[:, 1:])
    )

    # ── UNWRAPPING ───────────────────────────────────────────────────
    turn_xy = np.degrees(np.unwrap(np.radians(turn_xy_raw)))
    turn_yz = np.degrees(np.unwrap(np.radians(turn_yz_raw)))

    # ── Frame + time alignment ───────────────────────────────────────
    frames = frames_all[1:-1]
    time   = frames / FPS

    # ── SAFETY CHECK ─────────────────────────────────────────────────
    if not (len(frames) == len(turn_3d) == len(turn_xy) == len(turn_yz)):
        print("  ⚠ Length mismatch — skipping")
        continue

    # ── SAVE UNTRIMMED CSV ───────────────────────────────────────────
    out_df = pd.DataFrame({
        "frame": frames,
        "time_sec": time,

        "turning_angle_3d_deg": turn_3d,

        "turning_angle_xy_deg": turn_xy_raw,
        "turning_angle_yz_deg": turn_yz_raw,

        "turning_angle_xy_unwrapped_deg": turn_xy,
        "turning_angle_yz_unwrapped_deg": turn_yz,

        "turning_velocity_deg_s": turn_3d / DT,
    })

    out_path = os.path.join(OUTPUT_DIR, f"{trial}_TURNING_ANGLE_1000fps.csv")
    out_df.to_csv(out_path, index=False)

    print(f"  → Saved: {out_path}")

    # ── SAVE TRIMMED CSV ─────────────────────────────────────────────
    TRIM = 15   # ~15 ms at 1000 fps

    if len(out_df) > 2 * TRIM:
        trimmed_df = out_df.iloc[TRIM:-TRIM].copy()

        trim_out_path = os.path.join(
            OUTPUT_DIR,
            f"{trial}_TURNING_ANGLE_1000fps_TRIMMED.csv"
        )

        trimmed_df.to_csv(trim_out_path, index=False)

        print(f"  → Saved TRIMMED: {trim_out_path}")

    else:
        print("  ⚠ Too short to trim, skipping trimmed save")

print("\n✅ DONE: Centered turning angles (1000 fps, unwrapped, aligned + trimmed)")

Processing: Trial2_180rpmxyzpts_SUB_1000fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based_1000fps\Trial2_180rpmxyzpts_TURNING_ANGLE_1000fps.csv
  → Saved TRIMMED: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based_1000fps\Trial2_180rpmxyzpts_TURNING_ANGLE_1000fps_TRIMMED.csv
Processing: Trial2_200rpmxyzpts_SUB_1000fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based_1000fps\Trial2_200rpmxyzpts_TURNING_ANGLE_1000fps.csv
  → Saved TRIMMED: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based_1000fps\Trial2_200rpmxyzpts_TURNING_ANGLE_1000fps_TRIMMED.csv
Processing: Trial3_180rpmxyzpts_SUB_1000fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based_1000fps\Trial3_180rpmxyzpts_TURNING_ANGLE_1000fps.csv
  → Saved TRIMMED: ./../../dataFolders/MuscaChasingBea

In [9]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import Normalize
import plotly.graph_objects as go

# ── Config ─────────────────────────────────────────────────────────────
TURN_DIR = r"./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based_1000fps"
SUB_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled_1000fps"

FIG_DIR  = r"./../../dataFolders/MuscaChasingBeads/figures/TurningAnglePlots/subsample_turning_angle_center-based_1000fps"
HTML_OUT = r"./../../dataFolders/MuscaChasingBeads/figures/TurningAnglePlots/ALL_TRIALS_3D_1000fps.html"

os.makedirs(FIG_DIR, exist_ok=True)

# ── Get unique trials ─────────────────────────────────────────────────
all_files = glob.glob(os.path.join(TURN_DIR, "*_TURNING_ANGLE_1000fps.csv"))

trial_map = {}
for f in all_files:
    fname = os.path.basename(f)
    trial = fname.split("_TURNING_ANGLE_1000fps")[0]
    if trial not in trial_map:
        trial_map[trial] = f

turn_files = list(trial_map.values())

if not turn_files:
    raise FileNotFoundError("No turning angle files found")

# ── Plotly setup ─────────────────────────────────────────────────────
fig3d = go.Figure()
valid_trials = []

# ── MAIN LOOP ────────────────────────────────────────────────────────
for path in turn_files:
    fname = os.path.basename(path)
    trial = fname.replace("_TURNING_ANGLE_1000fps.csv", "")

    print(f"Processing: {trial}")

    df = pd.read_csv(path)
    time = df["time_sec"].values

    # ── 2D PLOTS ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    fig.suptitle(f"{trial} (1000 fps)", fontsize=12)

    for ax, col, label, color in zip(
        axes,
        ["turning_angle_3d_deg", "turning_angle_xy_deg", "turning_angle_yz_deg"],
        ["3D turning angle",     "XY turning angle",     "YZ turning angle"],
        ["steelblue",            "darkorange",           "seagreen"]
    ):
        ax.plot(time, df[col].values, lw=0.8, color=color)
        ax.set_ylabel("degrees")
        ax.set_title(label)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("time (s)")
    plt.tight_layout()

    fig.savefig(os.path.join(FIG_DIR, f"{trial}_turning_angle_1000fps.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

    print("  → Saved 2D plot")

    # ── 3D DATA ──────────────────────────────────────────────────────
    sub_path = glob.glob(os.path.join(SUB_DIR, f"{trial}*_SUB_1000fps.csv"))
    if not sub_path:
        print(f"  ⚠ No SUB file for {trial}")
        continue

    df_sub = pd.read_csv(sub_path[0])

    head = df_sub[["pt2_X", "pt2_Y", "pt2_Z"]].values
    turn = df["turning_angle_3d_deg"].values

    min_len = min(len(head), len(turn))
    head = head[:min_len]
    turn = turn[:min_len]

    x, y, z = head[:, 0], head[:, 1], head[:, 2]

    # ── CLEAN ────────────────────────────────────────────────────────
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & np.isfinite(turn)
    x, y, z, turn = x[valid], y[valid], z[valid], turn[valid]

    if len(x) < 2:
        print(f"  ⚠ Skipping {trial}")
        continue

    # ── STATIC 3D ────────────────────────────────────────────────────
    fig_static = plt.figure(figsize=(7, 6))
    ax3d = fig_static.add_subplot(111, projection='3d')

    norm = Normalize(vmin=np.min(turn), vmax=np.max(turn))
    cmap = cm.get_cmap('turbo')

    for i in range(len(x) - 1):
        ax3d.plot(
            x[i:i+2], y[i:i+2], z[i:i+2],
            color=cmap(norm(turn[i])),
            linewidth=2
        )

    mappable = cm.ScalarMappable(norm=norm, cmap=cmap)
    mappable.set_array(turn)
    cbar = fig_static.colorbar(mappable, ax=ax3d, pad=0.1)
    cbar.set_label("Turning Angle (°)")

    ax3d.set_title(f"{trial} — 3D Trajectory (1000 fps)")
    ax3d.set_xlabel("X")
    ax3d.set_ylabel("Y")
    ax3d.set_zlabel("Z")

    fig_static.savefig(
        os.path.join(FIG_DIR, f"{trial}_3D_turning_1000fps.png"),
        dpi=200,
        bbox_inches="tight"
    )
    plt.close(fig_static)

    print("  → Saved static 3D plot")

    # ── INTERACTIVE 3D ───────────────────────────────────────────────
    visible = True if len(valid_trials) == 0 else False

    fig3d.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines+markers",
        visible=visible,
        marker=dict(size=3, color=turn, colorscale="Turbo"),
        line=dict(color=turn, colorscale="Turbo", width=3),
        name=trial
    ))

    valid_trials.append(trial)

# ── DROPDOWN ─────────────────────────────────────────────────────────
buttons = []
n_traces = len(valid_trials)

for i, trial in enumerate(valid_trials):
    visibility = [False] * n_traces
    visibility[i] = True

    buttons.append(dict(
        label=trial,
        method="update",
        args=[{"visible": visibility},
              {"title": f"{trial} — 3D Trajectory (1000 fps)"}]
    ))

fig3d.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0.02,
        y=1.05
    )],
    title="3D Trajectory Viewer (1000 fps)",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=900,
    height=700
)

fig3d.write_html(HTML_OUT, include_plotlyjs=True, full_html=True)

print("\n✅ DONE (1000 fps)")
print(f"→ Plots: {FIG_DIR}")
print(f"→ Interactive: {HTML_OUT}")

Processing: Trial2_180rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_21252\3975380061.py:103: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial2_200rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_21252\3975380061.py:103: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial3_180rpmxyzpts
  → Saved 2D plot


...

  → Saved static 3D plot
Processing: Trial5_400rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_21252\3975380061.py:103: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial7_400rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_21252\3975380061.py:103: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot

✅ DONE (1000 fps)
→ Plots: ./../../dataFolders/MuscaChasingBeads/figures/TurningAnglePlots/subsample_turning_angle_center-based_1000fps
→ Interactive: ./../../dataFolders/MuscaChasingBeads/figures/TurningAnglePlots/ALL_TRIALS_3D_1000fps.html


In [10]:
# Removing edge effect (2d and 3d plots) 
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ── PATHS (1000 fps) ──────────────────────────────────────────────────
TURN_DIR = r"./../../dataFolders/MuscaChasingBeads/turning_angle/subsample_turning_angle_center-based_1000fps"
SUB_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled_1000fps"

OUT_3D_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based_1000fps/interactive_3D_cleaned"
OUT_2D_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/subsample_turning_angle_center-based_1000fps/plots_2D_cleaned"

os.makedirs(OUT_3D_DIR, exist_ok=True)
os.makedirs(OUT_2D_DIR, exist_ok=True)

# ── GET UNIQUE FILES ──────────────────────────────────────────────────
all_files = glob.glob(os.path.join(TURN_DIR, "*_TURNING_ANGLE_1000fps.csv"))

trial_map = {}
for f in all_files:
    fname = os.path.basename(f)
    trial = fname.split("_TURNING_ANGLE_1000fps")[0]
    if trial not in trial_map:
        trial_map[trial] = f

turn_files = list(trial_map.values())

if not turn_files:
    raise FileNotFoundError("No turning angle files found")

# ── MAIN LOOP ─────────────────────────────────────────────────────────
for path in turn_files:
    fname = os.path.basename(path)
    trial = fname.replace("_TURNING_ANGLE_1000fps.csv", "")

    print(f"\nProcessing: {trial}")

    df = pd.read_csv(path)

    # ── MATCH SUB FILE ────────────────────────────────────────────────
    sub_path = glob.glob(os.path.join(SUB_DIR, f"{trial}*_SUB_1000fps.csv"))
    if not sub_path:
        print("  ⚠ No SUB file found")
        continue

    df_sub = pd.read_csv(sub_path[0])

    # ── EXTRACT DATA ─────────────────────────────────────────────────
    head = df_sub[["pt2_X", "pt2_Y", "pt2_Z"]].values

    turn_3d = df["turning_angle_3d_deg"].values
    turn_xy = df["turning_angle_xy_deg"].values
    turn_yz = df["turning_angle_yz_deg"].values
    time    = df["time_sec"].values

    # ── ALIGN LENGTHS ─────────────────────────────────────────────────
    min_len = min(len(head), len(turn_3d))
    head    = head[:min_len]
    turn_3d = turn_3d[:min_len]
    turn_xy = turn_xy[:min_len]
    turn_yz = turn_yz[:min_len]
    time    = time[:min_len]

    x, y, z = head[:, 0], head[:, 1], head[:, 2]

    # ── CLEAN NaNs ───────────────────────────────────────────────────
    valid = (
        np.isfinite(x) & np.isfinite(y) & np.isfinite(z) &
        np.isfinite(turn_3d) & np.isfinite(turn_xy) & np.isfinite(turn_yz)
    )

    x, y, z = x[valid], y[valid], z[valid]
    turn_3d = turn_3d[valid]
    turn_xy = turn_xy[valid]
    turn_yz = turn_yz[valid]
    time    = time[valid]

    if len(x) < 10:
        print("  ⚠ Too few valid points")
        continue

    # ── REMOVE EDGE EFFECT ───────────────────────────────────────────
    x = x[15:-15]
    y = y[15:-15]
    z = z[15:-15]

    turn_3d = turn_3d[15:-15]
    turn_xy = turn_xy[15:-15]
    turn_yz = turn_yz[15:-15]
    time    = time[15:-15]

    if len(x) < 2:
        print("  ⚠ Too short after trimming")
        continue

    # ── 📈 2D PLOTS ──────────────────────────────────────────────────
    fig2d, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

    axes[0].plot(time, turn_3d, lw=1)
    axes[0].set_title("3D Turning Angle")
    axes[0].set_ylabel("deg")

    axes[1].plot(time, turn_xy, lw=1)
    axes[1].set_title("XY Turning Angle")
    axes[1].set_ylabel("deg")

    axes[2].plot(time, turn_yz, lw=1)
    axes[2].set_title("YZ Turning Angle")
    axes[2].set_ylabel("deg")
    axes[2].set_xlabel("time (s)")

    for ax in axes:
        ax.grid(True, alpha=0.3)

    fig2d.suptitle(f"{trial} — Edge-trimmed (1000 fps)")
    fig2d.tight_layout()

    out_2d = os.path.join(OUT_2D_DIR, f"{trial}_turning_2D_cleaned_1000fps.png")
    fig2d.savefig(out_2d, dpi=150)
    plt.close(fig2d)

    print("  → Saved 2D plot")

    # ── 🌐 3D INTERACTIVE ─────────────────────────────────────────────
    fig3d = go.Figure()

    fig3d.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines+markers",
        marker=dict(
            size=3,
            color=turn_3d,
            colorscale="Turbo",
            colorbar=dict(title="Turning Angle (°)")
        ),
        line=dict(color=turn_3d, colorscale="Turbo", width=4),
        name=trial
    ))

    fig3d.update_layout(
        title=f"{trial} — 3D Trajectory (1000 fps, edge-trimmed)",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z"
        ),
        width=900,
        height=700
    )

    out_3d = os.path.join(OUT_3D_DIR, f"{trial}_3D_cleaned_1000fps.html")
    fig3d.write_html(out_3d, include_plotlyjs=True, full_html=True)

    print("  → Saved 3D interactive")

print("\n✅ DONE: 1000 fps 2D + 3D plots generated")


Processing: Trial2_180rpmxyzpts
  → Saved 2D plot
  → Saved 3D interactive

Processing: Trial2_200rpmxyzpts
  → Saved 2D plot
  → Saved 3D interactive

Processing: Trial3_180rpmxyzpts
  → Saved 2D plot
  → Saved 3D interactive

Processing: Trial4_400rpmxyzpts
  → Saved 2D plot
  → Saved 3D interactive

Processing: Trial5_180rpmxyzpts
  → Saved 2D plot
  → Saved 3D interactive

Processing: Trial5_400rpmxyzpts
  → Saved 2D plot
  → Saved 3D interactive

Processing: Trial7_400rpmxyzpts
  → Saved 2D plot
  → Saved 3D interactive

✅ DONE: 1000 fps 2D + 3D plots generated


# Error angle

In [5]:
import os
import glob
import numpy as np
import pandas as pd

# ── Config (1000 fps) ─────────────────────────────────────────────────
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled_1000fps"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled_1000fps"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS = 1000

# ── Helpers ───────────────────────────────────────────────────────────
def compute_error_angle_head(bead, head, center):
    v_heading = head - center
    v_bead    = bead - head

    dot   = np.sum(v_heading * v_bead, axis=1)
    cross = np.linalg.norm(np.cross(v_heading, v_bead), axis=1)

    return np.degrees(np.arctan2(cross, dot))  # 0–180

def compute_hori_error(bead, head, center):
    v_heading = (head - center)[:, :2]
    v_bead    = (bead - head)[:, :2]

    dot   = np.sum(v_heading * v_bead, axis=1)
    cross = v_heading[:,0]*v_bead[:,1] - v_heading[:,1]*v_bead[:,0]

    return np.degrees(np.arctan2(cross, dot))  # -180 → 180

def compute_ver_error(bead, head, center):
    v_heading = (head - center)[:, 1:3]
    v_bead    = (bead - head)[:, 1:3]

    dot   = np.sum(v_heading * v_bead, axis=1)
    cross = v_heading[:,0]*v_bead[:,1] - v_heading[:,1]*v_bead[:,0]

    return np.degrees(np.arctan2(cross, dot))  # -180 → 180

# ── Process ───────────────────────────────────────────────────────────
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SUB_1000fps.csv"))

for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace("_SUB_1000fps.csv", "")
    print(f"Processing: {fname}")

    df     = pd.read_csv(path)
    frames = df["frame"].values
    time   = frames / FPS  
    bead   = df[["pt1_X","pt1_Y","pt1_Z"]].values
    head   = df[["pt2_X","pt2_Y","pt2_Z"]].values
    center = df[["center_X","center_Y","center_Z"]].values

    # ── Distance ─────────────────────────────────────────────────────
    dist_head = np.linalg.norm(bead - head, axis=1)

    # ── Error angles ─────────────────────────────────────────────────
    err_head = compute_error_angle_head(bead, head, center)

    err_hori_raw = compute_hori_error(bead, head, center)
    err_vert_raw = compute_ver_error(bead, head, center)

    # ── UNWRAP (ONLY FOR SIGNED ANGLES) ──────────────────────────────
    err_hori_unwrapped = np.degrees(np.unwrap(np.radians(err_hori_raw)))
    err_vert_unwrapped = np.degrees(np.unwrap(np.radians(err_vert_raw)))

    # ── Output ───────────────────────────────────────────────────────
    out_df = pd.DataFrame({
        "frame": frames,
        "time_sec": time,   

        "dist_head_m": dist_head,

        "error_angle_head_deg": err_head,

        "error_angle_horiz_deg": err_hori_raw,
        "error_angle_horiz_unwrapped_deg": err_hori_unwrapped,

        "error_angle_vert_deg": err_vert_raw,
        "error_angle_vert_unwrapped_deg": err_vert_unwrapped,
    })

    out_path = os.path.join(OUTPUT_DIR, f"{trial}_CHASE_METRICS_1000fps.csv")
    out_df.to_csv(out_path, index=False)

    print(f"  → Saved: {out_path}")

print("\n DONE: Error angles computed for 1000 fps")

Processing: Trial2_180rpmxyzpts_SUB_1000fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled_1000fps\Trial2_180rpmxyzpts_CHASE_METRICS_1000fps.csv
Processing: Trial2_200rpmxyzpts_SUB_1000fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled_1000fps\Trial2_200rpmxyzpts_CHASE_METRICS_1000fps.csv
Processing: Trial3_180rpmxyzpts_SUB_1000fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled_1000fps\Trial3_180rpmxyzpts_CHASE_METRICS_1000fps.csv
Processing: Trial4_400rpmxyzpts_SUB_1000fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled_1000fps\Trial4_400rpmxyzpts_CHASE_METRICS_1000fps.csv
Processing: Trial5_180rpmxyzpts_SUB_1000fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled_1000

In [16]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import Normalize
import plotly.graph_objects as go

# ── Config (1000 fps) ─────────────────────────────────────────────────
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/ErrorAngle_DistFromBead/ErrorAngle_DistFromBead_Subsampled_1000fps"
SUB_DIR    = r"./../../dataFolders/MuscaChasingBeads/xyz_Subsampled_1000fps"

FIG_DIR    = r"./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled_1000fps"
HTML_OUT   = r"./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled_1000fps/ALL_TRIALS_3D_1000fps.html"

os.makedirs(FIG_DIR, exist_ok=True)

FPS = 1000

# ── Get unique trials (avoid duplicates) ───────────────────────────────
all_files = glob.glob(os.path.join(INPUT_DIR, "*_CHASE_METRICS_1000fps.csv"))

trial_map = {}
for f in all_files:
    fname = os.path.basename(f)
    trial = fname.split("_CHASE_METRICS")[0]
    if trial not in trial_map:
        trial_map[trial] = f

csv_files = list(trial_map.values())

if not csv_files:
    raise FileNotFoundError("No CHASE_METRICS_1000fps CSV files found")

# ── Plotly setup ───────────────────────────────────────────────────────
fig3d = go.Figure()
buttons = []
valid_trials = []

# ── MAIN LOOP ──────────────────────────────────────────────────────────
for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace("_CHASE_METRICS_1000fps.csv", "")

    print(f"Processing: {trial}")

    df = pd.read_csv(path)
    time = df["frame"].values / FPS

    # ── 2D PLOTS ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(trial, fontsize=12)

    for ax, col, label, color in zip(
        axes,
        ["error_angle_head_deg", "error_angle_horiz_deg", "error_angle_vert_deg", "dist_head_m"],
        ["3D error angle",       "Horizontal error (XY)", "Vertical error (YZ)",  "Distance head→bead"],
        ["steelblue",            "darkorange",             "seagreen",             "mediumpurple"]
    ):
        ax.plot(time, df[col].values, lw=0.8, color=color)
        ax.set_ylabel("deg" if "deg" in col else "m")
        ax.set_title(label)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("time (s)")
    plt.tight_layout()

    fig.savefig(os.path.join(FIG_DIR, f"{trial}_error_angle_1000fps.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

    print("  → Saved 2D plot")

    # ── 3D DATA ────────────────────────────────────────────────────────
    sub_path = glob.glob(os.path.join(SUB_DIR, f"{trial}*_SUB_1000fps.csv"))
    if not sub_path:
        print(f"  ⚠ No SUB file for {trial}")
        continue

    df_sub = pd.read_csv(sub_path[0])

    head = df_sub[["pt2_X", "pt2_Y", "pt2_Z"]].values
    err  = df["error_angle_head_deg"].values

    min_len = min(len(head), len(err))
    head = head[:min_len]
    err  = err[:min_len]

    x, y, z = head[:, 0], head[:, 1], head[:, 2]

    # ── CLEAN NaNs / Infs ──────────────────────────────────────────────
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & np.isfinite(err)
    x, y, z, err = x[valid], y[valid], z[valid], err[valid]

    if len(x) < 2:
        print(f"  ⚠ Skipping {trial} (insufficient valid data)")
        continue

    # ── STATIC 3D (COLORED LINE) ───────────────────────────────────────
    fig_static = plt.figure(figsize=(7, 6))
    ax3d = fig_static.add_subplot(111, projection='3d')

    norm = Normalize(vmin=np.min(err), vmax=np.max(err))
    cmap = cm.get_cmap('turbo')

    for i in range(len(x) - 1):
        color = cmap(norm(err[i]))
        ax3d.plot(
            x[i:i+2],
            y[i:i+2],
            z[i:i+2],
            color=color,
            linewidth=2
        )

    mappable = cm.ScalarMappable(norm=norm, cmap=cmap)
    mappable.set_array(err)
    cbar = fig_static.colorbar(mappable, ax=ax3d, pad=0.1)
    cbar.set_label("Error Angle (°)")

    ax3d.set_title(f"{trial} — 3D Trajectory (Error Angle)", fontsize=10)
    ax3d.set_xlabel("X")
    ax3d.set_ylabel("Y")
    ax3d.set_zlabel("Z")

    max_range = np.array([
        x.max()-x.min(),
        y.max()-y.min(),
        z.max()-z.min()
    ]).max() / 2.0

    if max_range == 0:
        max_range = 1e-6

    mid_x = (x.max()+x.min()) * 0.5
    mid_y = (y.max()+y.min()) * 0.5
    mid_z = (z.max()+z.min()) * 0.5

    ax3d.set_xlim(mid_x - max_range, mid_x + max_range)
    ax3d.set_ylim(mid_y - max_range, mid_y + max_range)
    ax3d.set_zlim(mid_z - max_range, mid_z + max_range)

    ax3d.view_init(elev=20, azim=135)

    fig_static.savefig(
        os.path.join(FIG_DIR, f"{trial}_3D_error_1000fps.png"),
        dpi=200,
        bbox_inches="tight"
    )
    plt.close(fig_static)

    print("  → Saved static 3D plot")

    # ── INTERACTIVE 3D (FIXED COLORBAR) ────────────────────────────────
    visible = True if len(valid_trials) == 0 else False

    fig3d.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines+markers",
        visible=visible,

        marker=dict(
            size=3,
            color=err,
            colorscale="Turbo",
            colorbar=dict(title="Error Angle (°)")
        ),

        line=dict(
            color=err,
            colorscale="Turbo",
            width=3
        ),

        name=trial
    ))

    valid_trials.append(trial)

# ── Dropdown ───────────────────────────────────────────────────────────
n_traces = len(valid_trials)

for i, trial in enumerate(valid_trials):
    visibility = [False] * n_traces
    visibility[i] = True

    buttons.append(dict(
        label=trial,
        method="update",
        args=[{"visible": visibility},
              {"title": f"{trial} — 3D Trajectory (Error Angle)"}]
    ))

fig3d.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0.02,
        y=1.05
    )],
    title="3D Error Angle Trajectory Viewer (1000 fps)",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=900,
    height=700,
    scene_camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
)

fig3d.write_html(HTML_OUT, include_plotlyjs=True, full_html=True)

print("\n DONE")
print(f"→ 2D plots: {FIG_DIR}")
print(f"→ Static 3D plots: {FIG_DIR}")
print(f"→ Interactive HTML: {HTML_OUT}")

Processing: Trial2_180rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_21252\3537438124.py:106: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial2_200rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_21252\3537438124.py:106: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial3_180rpmxyzpts
  → Saved 2D plot


...

  → Saved static 3D plot
Processing: Trial5_400rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_21252\3537438124.py:106: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot
Processing: Trial7_400rpmxyzpts
  → Saved 2D plot


C:\Users\munpa\AppData\Local\Temp\ipykernel_21252\3537438124.py:106: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



  → Saved static 3D plot

✅ DONE
→ 2D plots: ./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled_1000fps
→ Static 3D plots: ./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled_1000fps
→ Interactive HTML: ./../../dataFolders/MuscaChasingBeads/figures/ChaseMetrics/ErrorAngle_DistFromBead_Subsampled_1000fps/ALL_TRIALS_3D_1000fps.html


# Speed

In [11]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample

# ── PATHS ─────────────────────────────────────────────────────────────
INPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Speed"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_1000fps"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS_ORIG = 2500
FPS_NEW  = 1000

# ── LOOP ─────────────────────────────────────────────────────────────
csv_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))

for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace(".csv", "")

    print(f"\nProcessing (1000 fps): {trial}")

    df = pd.read_csv(path)

    if "speed" not in df.columns:
        print("  ⚠ No speed column")
        continue

    speed = df["speed"].values

    # ── RESAMPLE ─────────────────────────────────────────────────────
    n_new = int(len(speed) * FPS_NEW / FPS_ORIG)
    speed_1000 = resample(speed, n_new)

    # ── SAVE ─────────────────────────────────────────────────────────
    out_df = pd.DataFrame({
        "frame": np.arange(len(speed_1000)),
        "speed": speed_1000
    })

    out_path = os.path.join(OUTPUT_DIR, f"{trial}_SPEED_1000fps.csv")
    out_df.to_csv(out_path, index=False)

    print(f"  → Saved: {out_path}")

    # ── TIME ─────────────────────────────────────────────────────────
    t_orig  = np.arange(len(speed)) / FPS_ORIG
    t_1000  = np.arange(len(speed_1000)) / FPS_NEW

print("\n DONE: 1000 fps speed")


Processing (1000 fps): Trial2_180rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_1000fps\Trial2_180rpmxyzpts_SPEED_SPEED_1000fps.csv

Processing (1000 fps): Trial2_200rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_1000fps\Trial2_200rpmxyzpts_SPEED_SPEED_1000fps.csv

Processing (1000 fps): Trial3_180rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_1000fps\Trial3_180rpmxyzpts_SPEED_SPEED_1000fps.csv

Processing (1000 fps): Trial4_400rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_1000fps\Trial4_400rpmxyzpts_SPEED_SPEED_1000fps.csv

Processing (1000 fps): Trial5_180rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_1000fps\Trial5_180rpmxyzpts_SPEED_SPEED_1000fps.csv

Processing (1000 fps): Trial5_400rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_1000fps\

In [13]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── PATHS ─────────────────────────────────────────────────────────────
INPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Speed/Speed_Subsampled_1000fps"

OUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/Speed/1000fps"
os.makedirs(OUT_DIR, exist_ok=True)

FPS = 1000

# ── LOOP ─────────────────────────────────────────────────────────────
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_1000fps.csv"))

if not csv_files:
    raise FileNotFoundError("No 1000 fps speed files found")

for path in csv_files:
    fname = os.path.basename(path)
    trial = fname.replace("_SPEED_1000fps.csv", "")

    print(f"Processing: {trial}")

    df = pd.read_csv(path)

    if "speed" not in df.columns:
        print("  ⚠ No speed column")
        continue

    speed = df["speed"].values

    # ── TIME AXIS ────────────────────────────────────────────────────
    time = np.arange(len(speed)) / FPS

    # ── CLEAN NaNs ───────────────────────────────────────────────────
    valid = np.isfinite(speed)
    speed = speed[valid]
    time  = time[valid]

    if len(speed) < 10:
        print("  ⚠ Too few valid points")
        continue

    # ── PLOT ─────────────────────────────────────────────────────────
    plt.figure(figsize=(10,5))

    plt.plot(time, speed, lw=1.5)

    plt.title(f"{trial} — Speed (1000 fps)")
    plt.xlabel("Time (s)")
    plt.ylabel("Speed")
    plt.grid(True)

    plt.tight_layout()

    # ── SAVE ─────────────────────────────────────────────────────────
    out_path = os.path.join(OUT_DIR, f"{trial}_speed_1000fps.png")
    plt.savefig(out_path, dpi=150)
    plt.close()

    print(f"  → Saved: {out_path}")

print("\n DONE: All 1000 fps speed plots saved")

Processing: Trial2_180rpmxyzpts_SPEED
  ⚠ Too few valid points
Processing: Trial2_200rpmxyzpts_SPEED
  ⚠ Too few valid points
Processing: Trial3_180rpmxyzpts_SPEED
  ⚠ Too few valid points
Processing: Trial4_400rpmxyzpts_SPEED
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Speed/1000fps\Trial4_400rpmxyzpts_SPEED_speed_1000fps.png
Processing: Trial5_180rpmxyzpts_SPEED
  ⚠ Too few valid points
Processing: Trial5_400rpmxyzpts_SPEED
  ⚠ Too few valid points
Processing: Trial7_400rpmxyzpts_SPEED
  ⚠ Too few valid points

✅ DONE: All 1000 fps speed plots saved
